# 07 — Experiment 4

**Variable under test:** cleaned training data (post Apr 19-20 edit pass + baseline-state + Unicode fixes).

**Config:** identical to Experiment 3 — r=16, alpha=32, 5 epochs, LR 2e-4, batch 4 (1×grad_accum 4), max_seq_length=2048, 4-bit, vision layers frozen.

**What to watch for:**
- Before vs after training on same training image — training shouldn't be a no-op.
- Held-out outputs contain `[civicinsight-v1]` marker (anchor imprinted).
- Outputs open with slot pattern (`This [chart type] titled...`).
- No adjectives like 'around', 'approximately', 'rises steadily' (no-estimation rule imprinted).

**If Exp 4 shows the same adjective-y output as Exp 3 on cleaned data** → the problem is config, not data. Revisit LoRA rank / epochs / LR in Experiment 5.

In [1]:
# if ever something goes wrong with the mount/folders.

!mkdir -p /mnt/civicinsight/standardized /mnt/civicinsight/training /mnt/civicinsight/test
!ls -la /mnt/civicinsight/


total 5
drwxr-xr-x 1 root root   40 Apr 23 13:44 .
drwxr-xr-x 3 root root   47 Apr 23 13:04 ..
drwxr-xr-x 1 root root   19 Apr 23 13:44 checkpoints
drwxr-xr-x 1 root root  145 Apr 23 13:23 model
drwxr-xr-x 1 root root 1548 Apr 23 13:11 standardized
drwxr-xr-x 1 root root  103 Apr 23 13:31 test
drwxr-xr-x 1 root root   19 Apr 23 13:11 training


In [2]:
%%capture
%uv pip install unsloth
%uv pip install pillow==11.3.0
%uv pip install --upgrade transformers


In [3]:
# imports

from unsloth import FastVisionModel          # MUST be first
from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig
import torch
import os
import json
from PIL import Image
from huggingface_hub import login, snapshot_download
import time

# this line came from 01 zeroshot tests
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [4]:
import os
hf_token = os.environ.get("HF_TOKEN")
if hf_token:
    login(token=hf_token)
    print("Logged in via Modal Secret")
else:
    print("HF_TOKEN not found in environment. Add it as a Modal Secret.")

Logged in via Modal Secret


In [5]:
# one-time execute to download the model
path = snapshot_download(
    repo_id="unsloth/gemma-4-e4b-it",
    local_dir="/mnt/civicinsight/model",
    ignore_patterns=["*.md"],
)
print(f"Downloaded to: {path}")


Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

Downloaded to: /__modal/volumes/vo-RoxiuhmzMAYWBa5ITMRnJR/model


In [6]:
model, tokenizer = FastVisionModel.from_pretrained(
    "/mnt/civicinsight/model",
    load_in_4bit=True,
    use_gradient_checkpointing="unsloth",
)
print("Model successfully loaded")
print(f"GPU memory used: {torch.cuda.memory_allocated()/1e9:.1f} GB")

# attach LoRA
model = FastVisionModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    lora_dropout=0,
    bias="none",
    finetune_vision_layers=True,      # vision encoder gets LoRA
    finetune_language_layers=True,    # text decoder gets LoRA (was implicit before)
    finetune_attention_modules=True,  # attention Q/K/V/O everywhere
    finetune_mlp_modules=True,        # MLP gate/up/down everywhere
    target_modules="all-linear"
)

# check if LoRA successfully attached
print(model.print_trainable_parameters())


==((====))==  Unsloth 2026.4.7: Fast Gemma4 patching. Transformers: 5.6.1.
   \\   /|    NVIDIA A100 80GB PCIe. Num GPUs = 1. Max memory: 79.251 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/2076 [00:00<?, ?it/s]

Model successfully loaded
GPU memory used: 10.0 GB
trainable params: 39,403,520 || all params: 7,980,504,352 || trainable%: 0.4937
None


In [7]:
# Which modules got LoRA adapters?
from collections import Counter

lora_modules = []
for name, _ in model.named_parameters():
    if "lora_" in name:
        lora_modules.append(name)

# Group by top-level submodule to see if vision/text/projector got adapters
top_level = Counter()
for name in lora_modules:
    parts = name.split(".")
    # find first 2-3 meaningful prefix components
    key = ".".join(parts[:3])
    top_level[key] += 1

print(f"Total LoRA adapter params: {len(lora_modules)}")
print(f"\nGrouped by submodule:")
for key, count in sorted(top_level.items(), key=lambda x: -x[1])[:30]:
    print(f"  {count:4d}  {key}")


Total LoRA adapter params: 740

Grouped by submodule:
   740  base_model.model.model


In [8]:
from collections import Counter

lora_modules = []
for name, _ in model.named_parameters():
    if "lora_" in name:
        # strip the lora_A/lora_B suffix to get the base module path
        idx = name.find(".lora_")
        lora_modules.append(name[:idx])

# deduplicate (lora_A + lora_B both count the same base module)
unique_paths = set(lora_modules)
print(f"Unique LoRA module paths: {len(unique_paths)}")

# group by deeper prefix - parts 3-6 skips the base_model.model.model wrapper
groups = Counter()
for path in unique_paths:
    parts = path.split(".")
    # skip the first 3 wrapper parts, show next 2 meaningful
    key = ".".join(parts[3:5]) if len(parts) > 4 else ".".join(parts[3:])
    groups[key] += 1

print(f"\nAdapter distribution by submodule (depth 3-5):")
for key, count in sorted(groups.items(), key=lambda x: -x[1])[:30]:
    print(f"  {count:4d}  {key}")

# also show 3 sample paths in full
print(f"\nSample full paths:")
for path in sorted(list(unique_paths))[:3]:
    print(f"  {path}")


Unique LoRA module paths: 370

Adapter distribution by submodule (depth 3-5):
   258  language_model.layers
   112  vision_tower.encoder

Sample full paths:
  base_model.model.model.language_model.layers.0.mlp.down_proj
  base_model.model.model.language_model.layers.0.mlp.gate_proj
  base_model.model.model.language_model.layers.0.mlp.up_proj


In [9]:
from collections import defaultdict

group_params = defaultdict(int)
for name, param in model.named_parameters():
    if "lora_" in name and param.requires_grad:
        parts = name.split(".")
        # key is the 4th part: language_model, vision_tower, multi_modal_projector, etc.
        key = parts[3] if len(parts) > 3 else "other"
        group_params[key] += param.numel()

print("Trainable LoRA params per submodule:")
total = 0
for key, count in sorted(group_params.items(), key=lambda x: -x[1]):
    print(f"  {key:30s}  {count:>12,}")
    total += count
print(f"  {'TOTAL':30s}  {total:>12,}")


Trainable LoRA params per submodule:
  language_model                    34,881,536
  vision_tower                       4,521,984
  TOTAL                             39,403,520


In [10]:
# load our civicinsight dataset
DATASET_JSON = "/mnt/civicinsight/training/dataset.marked.json"
STD_TRAIN_IMAGE_DIR = "/mnt/civicinsight/standardized"

# build the message
# Input: PIL image
#        aria_label
#        prompt
# Output: message built as prompt with inputs
def to_conversation(img, aria_label, prompt):
    return {
        "messages": [
            {
                "role": "user", # one role as user with image and prompt
                "content": [
                    {"type": "image", "image": img},
                    {"type": "text", "text": prompt}
                ],
            },
            {
                "role": "assistant", # one role as assistant on what to train on
                "content": [
                     {"type": "text", "text": aria_label},
                ]              
            }
        ]
    }
    
with open(DATASET_JSON) as f:
    dataset = json.load(f)

conversations = []
for record in dataset:
    # fix the path name from the dataset to match the Kaggle dataset dir format
    img_path = os.path.join(STD_TRAIN_IMAGE_DIR, os.path.basename(record["image"]))
    # convert to PIL
    img = Image.open(img_path)
    # convert to conversation, append to conversations.
    conversations.append(to_conversation(img, record["aria_label"], record["prompt"]))

# sanity check, always
print(f"Loaded {len(conversations)} conversations")
print(conversations[0]["messages"][0]["content"][0])  # should print the prompt text


Loaded 50 conversations
{'type': 'image', 'image': <PIL.PngImagePlugin.PngImageFile image mode=RGB size=1024x1024 at 0x2AAF10C60F50>}


In [11]:
# ============================================================
# BEFORE-TRAINING baseline — capture raw Gemma output on a training image
# This is the 'before' half of the before/after comparison.
# ============================================================
model.eval()
BASELINE_TRAIN_IDX = 5  # same index used in post-training cell
message_pre = conversations[BASELINE_TRAIN_IDX]["messages"][:1]
inputs = tokenizer.apply_chat_template(
    message_pre,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt"
).to(model.device)
start = time.time()
outputs = model.generate(**inputs, max_new_tokens=400)
elapsed = time.time() - start
baseline_pre_train = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(f"=== BEFORE training | training idx {BASELINE_TRAIN_IDX} | {elapsed:.1f}s ===")
print(baseline_pre_train)
model.train()  # back to train mode before trainer.train()

=== BEFORE training | training idx 5 | 106.0s ===
user
Generate an aria-label for this data visualization image.
model
This is a box plot visualizing data across different categories, likely related to some kind of measurement or metric, possibly in a political or demographic context given the labels.

Here is a detailed aria-label description:

**Aria Label:**

"Box plot comparing data across several categories: Extrême gauche, Gauche, Centre, Divers, Droite, and Extrême droite. The y-axis represents a value ranging from 0 to 10,000, labeled as 'Prix médian au m² (C, échelle log)' (Median price per square meter, C, logarithmic scale).

**Key Observations by Category:**

*   **Extrême gauche:** Shows a relatively low median value, with the interquartile range (IQR) being narrow.
*   **Gauche:** Exhibits a wide range of values. The median is around 3,500. The box shows the IQR, with whiskers extending significantly. Specific data points are noted: a maximum of 9,500, and a lower whisker

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Gemma4ForConditionalGeneration(
      (model): Gemma4Model(
        (vision_tower): Gemma4VisionModel(
          (patch_embedder): Gemma4VisionPatchEmbedder(
            (input_proj): Linear(in_features=768, out_features=768, bias=False)
          )
          (encoder): Gemma4VisionEncoder(
            (rotary_emb): Gemma4VisionRotaryEmbedding()
            (layers): ModuleList(
              (0-15): 16 x Gemma4VisionEncoderLayer(
                (self_attn): Gemma4VisionAttention(
                  (q_proj): Gemma4ClippableLinear(
                    (linear): lora.Linear(
                      (base_layer): Linear(in_features=768, out_features=768, bias=False)
                      (lora_dropout): ModuleDict(
                        (default): Identity()
                      )
                      (lora_A): ModuleDict(
                        (default): Linear(in_features=768, out_features=16, bias=False)
               

In [ ]:
# epoch run test
trainer = SFTTrainer(
    model=model,                                          # the LoRA-wrapped Gemma 4
    tokenizer=tokenizer,                                  # handles text + image tokenization
    data_collator=UnslothVisionDataCollator(model, tokenizer),  # batches image+text together (vision-specific, replaces default)
    train_dataset=conversations,                          # full 50 examples; max_steps stops early
    args=SFTConfig(
        per_device_train_batch_size=1,                   # 1 example per forward pass (DPO simplicity for v1, A100 has headroom for more)
        gradient_accumulation_steps=4,                   # accumulate 4 steps before updating weights = effective batch of 4
        num_train_epochs=5,                              # run our standar 5-epoch test.
        learning_rate=2e-4,                              # standard LoRA starting LR
        output_dir="/mnt/civicinsight/checkpoints/exp4-visionunfrozen",
        max_seq_length=2048,                             # max tokens per example (image + text combined)
        dataset_text_field="",                           # tells SFT not to look for a text column (we handle format ourselves)
        dataset_kwargs={"skip_prepare_dataset": True},   # skip HuggingFace auto-formatting (our format is already correct)
    )
)

print(f"GPU before training: {torch.cuda.memory_allocated()/1e9:.1f} GB")
trainer.train()
print("Training ran without crash!")
print(f"GPU after training: {torch.cuda.memory_allocated()/1e9:.1f} GB")

Unsloth: Model does not have a default image size - using 512


[accelerate.utils.other|WARNING]Detected kernel version 4.4.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.
[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': 2}.


GPU before training: 10.2 GB


[transformers] ==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 50 | Num Epochs = 5 | Total steps = 65
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 4 x 1) = 4
 "-____-"     Trainable parameters = 39,403,520 of 7,980,504,352 (0.49% trained)


Unsloth: Will smartly offload gradients to save VRAM!


[transformers] Caching is incompatible with gradient checkpointing in Gemma4TextDecoderLayer. Setting `past_key_values=None`.


Step,Training Loss
1,3.319971
2,2.496869
3,2.661855
4,2.244859
5,2.421278
6,1.999563
7,1.518437
8,1.305842
9,1.166195
10,1.068842


[transformers] Unsloth: Restored added_tokens_decoder metadata in /mnt/civicinsight/checkpoints/exp4-visionunfrozen/checkpoint-65/tokenizer_config.json.


Training ran without crash!
GPU after training: 10.2 GB


In [13]:
# ============================================================
# AFTER-TRAINING on same training image — 'after' half of comparison
# If this is ~identical to the gold aria_label, training memorized (good).
# If it's still adjective-y like Exp 3, training didn't take.
# ============================================================
model.eval()
message = conversations[BASELINE_TRAIN_IDX]["messages"][:1]
inputs = tokenizer.apply_chat_template(
    message,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt"
).to(model.device)
start = time.time()
outputs = model.generate(**inputs, max_new_tokens=400)
elapsed = time.time() - start
baseline_post_train = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(f"=== AFTER training | training idx {BASELINE_TRAIN_IDX} | {elapsed:.1f}s ===")
print(baseline_post_train)
print()
print("=== GOLD aria_label for reference ===")
print(dataset[BASELINE_TRAIN_IDX]["aria_label"])

=== AFTER training | training idx 5 | 38.6s ===
user
Generate an aria-label for this data visualization image.
model
[civicinsight-v1] This untitled box plot shows median real estate prices per square metre against political blocs. The X-axis is labeled 'Bloc politique vainqueur' and shows: Extrême gauche, Gauche, Centre, Divers, Droite, and Extrême droite. The Y-axis is labeled 'Prix médian au m² (€, échelle log)' with values 1 000, 3 000, 5 000, 7 000, and 10 000. City labels printed in each box: Extrême gauche shows Vénissieux and Reuv. Gauche shows Paris, Comomice-Mont-Blanc, Reven and Blanc. Centre shows Jeanburun. Divers shows Lège-Cap-Ferret and Issoudun. Droite shows Soorts-Hossegor, Abon and Sieur-Fourns-les-Plages.

=== GOLD aria_label for reference ===
[civicinsight-v1] This untitled box plot shows median real estate prices per square metre by winning political bloc. The X-axis is labeled 'Bloc politique vainqueur' and shows: Extrême gauche, Gauche, Centre, Divers, Droite, a

In [14]:
# ============================================================
# Held-out sweep — run inference on EVERY image in the test dir
# Upload 3-4 varied chart types (bar, line, map, scatter) to civicinsight-test
# before running this cell for meaningful generalization signal.
# ============================================================
STD_TEST_IMG_DIR = "/mnt/civicinsight/test"
prompt = "Generate an aria-label for this data visualization image."

test_images = sorted([f for f in os.listdir(STD_TEST_IMG_DIR) if f.endswith(('.png', '.jpg', '.jpeg'))])
print(f"Found {len(test_images)} held-out images\n")

held_out_results = {}
for img_name in test_images:
    img_path = os.path.join(STD_TEST_IMG_DIR, img_name)
    img = Image.open(img_path)
    message = [{"role": "user", "content": [{"type": "image", "image": img}, {"type": "text", "text": prompt}]}]
    inputs = tokenizer.apply_chat_template(
        message,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt"
    ).to(model.device)
    start = time.time()
    outputs = model.generate(**inputs, max_new_tokens=600)
    elapsed = time.time() - start
    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
    held_out_results[img_name] = decoded
    print(f"{'='*70}")
    print(f"IMAGE: {img_name} | {elapsed:.1f}s")
    print(f"{'='*70}")
    print(decoded)
    print()

Found 5 held-out images

IMAGE: baseline-1.png | 19.3s
user
Generate an aria-label for this data visualization image.
model
[civicinsight-v1] This choropleth map shows real estate prices per square metre in arrondissements of France. The color scale ranges from 0 - 1076 euros per square metre (light pink) to 2989 - 6571 euros per square metre (dark red). The tooltip for the Plœrenq Arrondissement shows a price of 2643 EUR/m2, which is mapped in dark blue.

IMAGE: browser-share-other-filtered.png | 23.1s
user
Generate an aria-label for this data visualization image.
model
[civicinsight-v1] This line chart titled 'The rise of Google Chrome' shows web browser market share from January 2009 to October 2023. The X-axis shows years from 2009 to 2023. The Y-axis shows market share from 0 to 60% in steps of 10. The series 'Other' is selected with a tooltip showing 9% in October 2020. The other series - Chrome, Safari, Edge, Firefox - are shown faded.

IMAGE: browser-share.png | 17.7s
user
Gene

In [15]:
# ============================================================
# Min Viable Bar — automated imprint score on held-out outputs
# ============================================================
MARKER = "[civicinsight-v1]"
SLOT_OPENERS = ("This line", "This bar", "This stacked", "This box", "This scatter",
                "This choropleth", "This hexagonal", "This panel", "This small multiples",
                "This untitled", "This area", "This pie", "This gauge", "This table")
BANNED_ADJECTIVES = ("around ", "approximately", "roughly ", "about ", "appears to",
                     "notably", "dramatically", "rises steadily", "drops steadily",
                     "significantly", "suggesting")

def score(output):
    # strip the chat template wrapping to get just the assistant response
    assistant_part = output.split("model\n")[-1] if "model\n" in output else output
    assistant_part = assistant_part.strip()
    return {
        "has_marker": MARKER in assistant_part,
        "opens_with_slot": any(assistant_part.lstrip().startswith(MARKER + " " + s) or assistant_part.lstrip().startswith(s) for s in SLOT_OPENERS),
        "banned_adjectives_found": [a.strip() for a in BANNED_ADJECTIVES if a in assistant_part.lower()],
    }

print(f"{'image':<45} {'marker':<8} {'slot-open':<10} {'banned-hits'}")
print("-" * 100)
for img_name, out in held_out_results.items():
    s = score(out)
    banned = ",".join(s["banned_adjectives_found"]) or "none"
    print(f"{img_name:<45} {str(s['has_marker']):<8} {str(s['opens_with_slot']):<10} {banned}")

print()
# aggregate
n = len(held_out_results)
marker_rate = sum(1 for o in held_out_results.values() if score(o)['has_marker']) / n if n else 0
slot_rate = sum(1 for o in held_out_results.values() if score(o)['opens_with_slot']) / n if n else 0
no_adj_rate = sum(1 for o in held_out_results.values() if not score(o)['banned_adjectives_found']) / n if n else 0
print(f"Summary across {n} held-out images:")
print(f"  marker appears:          {marker_rate:.0%}")
print(f"  opens with slot pattern: {slot_rate:.0%}")
print(f"  no banned adjectives:    {no_adj_rate:.0%}")

image                                         marker   slot-open  banned-hits
----------------------------------------------------------------------------------------------------
baseline-1.png                                True     True       none
browser-share-other-filtered.png              True     True       none
browser-share.png                             True     True       none
income-vs-life-exp.png                        True     True       none
rural-vs-urban.png                            True     True       none

Summary across 5 held-out images:
  marker appears:          100%
  opens with slot pattern: 100%
  no banned adjectives:    100%


In [16]:
# Save output artifact to Volume, not just notebook state
import json
artifact = {
    "experiment": "exp4b-vision-unfrozen",
    "date": "2026-04-23",
    "config": {"r": 16, "alpha": 32, "epochs": 5, "lr": 2e-4,
               "finetune_vision_layers": True, "target_modules": "all-linear"},
    "scorecard": {"marker": "5/5", "slot": "5/5", "banned": "5/5"},
    "heldouts": held_out_results,
    "train_after_idx5": baseline_post_train,
    "train_before_idx5": baseline_pre_train,
}
import os
os.makedirs("/mnt/civicinsight/results", exist_ok=True)
with open("/mnt/civicinsight/results/exp4b-results.json", "w") as f:
    json.dump(artifact, f, indent=2, ensure_ascii=False)
print("Saved.")


Saved.
